# Full-Range Data Augmentation (0.1 – 100 µM)

This notebook extends the logic started in the `data_augumentation.ipynb` notebook for the 7.5 - 20 µM range, considering the non linearity after the 20 µM concentration treshold because of the suprasaturation.

In versiunea initiala, am folosit o interpolare liniara intre anchor curves (justificata de ecuația Randles-Ševčík) si zgomotul Gaussian cu σ constant de 0.001 µA era o alegere rezonabila pentru interval.

Extinderea la intervalul complet 0.1–100 µM a scos la suprafata mai multe probleme care, în versiunea restrânsă, fie nu existau, fie nu se vedeau.

## Key elements that went under review and suffered changes



| Problemă | Cauza | Fix aplicat |
|---|---|---|
| initial, `S8_all_default` avea o performanță mai slabă decât intervalul limitat din `data_augumentation.ipynb` | Lorentzian σ la 0.1–0.5 µM e mult mai mare decât zgomotul real Long & Winefordner, îngroapă peak-ul | Înlocuit profilul Lorentzian cu modelul **Long & Winefordner (L&W)** ajustat pe replicate reale; am adăugat SNR floor |
| Oversampling invers-log strică vs uniform | Cu σ extrem la lower-end, supra-reprezentarea 0.1–1 µM adauga semnale fantoma cu zgomot mare | Fix-ul σ L&W rezolvă în mare; boost redus de la 3× la 2× pentru ajutor |
| Baseline polinomial la concentrații mari | Amplitudinea 0.05 µA e ~5–15% din peak-ul de 0.1 µM distorsiuni fizic implausibile | Amplitudinea baseline scalează acum cu concentrația: `amp(c) = amp_max × tanh(c / c_ref)` |
| `DecisionTree` în ablation study | DT cu depth 5 tot memorează cele 39 de puncte reale → MAE ≈ 0 pe Protocol A, strică scala graficelor | **Scos din ablation** |
| Sampling uniform în segment (concentrații) | `np.random.uniform(c_lo, c_hi)` supra-reprezintă zona înaltă pe scala log | Acum se trage **log-uniform**: `exp(U(log c_lo, log c_hi))` |


### Strategia recomandata:

`S_REFINED` = PCHIP interpolation + L&W noise + concentration-scaled baseline + potential drift + mild log-stratified oversampling (boost=2).  
This replaces `S8_all_default` as the default recommended configuration.


---
## 0 · Imports and Global Configuration

In [2]:
# for root anchor (notebook moved into subfolder)
import sys
from pathlib import Path
ROOT_DIR = Path().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator
from scipy.signal import savgol_filter
from scipy.stats import linregress

# Plotly for interactive visualisation
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Custom plot function
from plot_style import (
    apply_default_plotly_layout,
    PLOTLY_TEMPLATE
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# project-specific classes
from voltammogram_signal import Signal
from peak import Peak

# centralized paths for all project files
import paths

warnings.filterwarnings('ignore')

GLOBAL_RANDOM_SEED = 42
np.random.seed(GLOBAL_RANDOM_SEED)


# custom color palette for plotting anchors
ANCHOR_COLOR_PALETTE = [
    '#2c0e91','#3b0fa3','#5a23c4','#7b3fd6','#9b5fe0',
    '#b07ddb','#c79bd2','#e4b0a8','#ee9070','#f06d3a',
    '#e34a1a','#b51d00',
]

print('Imports OK.')

Imports OK.


---
## 1 · Load Data

In [3]:
raw_calibration_dataframe = pd.read_excel(paths.CALIBRATION_WORKBOOK, sheet_name='Raw data')

potential_grid_V          = raw_calibration_dataframe.iloc[1:, 0].values.astype(float)
blank_baseline_current_uA = raw_calibration_dataframe.iloc[1:, 1].values.astype(float)
raw_signal_matrix_uA      = raw_calibration_dataframe.iloc[1:, 2:].values.astype(float)

CONCENTRATIONS = [
    100, 100, 100,
     50,  50,  50,
     25,  25,  25,
     15,  15,  15,
     10,  10,  10,
    7.5, 7.5, 7.5,
      5,   5,   5,
    2.5, 2.5,
      1,   1,   1,
    0.5, 0.5, 0.5, 0.5, 0.5, 0.5,
   0.25, 0.25, 0.25,
    0.1,  0.1,  0.1,  0.1,  0.1,
]

ANCHOR_CONCENTRATIONS = [0.1, 0.25, 0.5, 1.0, 2.5, 5.0, 7.5, 10.0, 15.0, 25.0, 50.0, 100.0]

print(f'Potential range  : {potential_grid_V.min():.4f} V  →  {potential_grid_V.max():.4f} V')
print(f'Number of points : {len(potential_grid_V)}')
print(f'Signal columns   : {raw_signal_matrix_uA.shape[1]}')
print(f'Anchor concentrations: {ANCHOR_CONCENTRATIONS}')

Potential range  : -0.6001 V  →  0.5023 V
Number of points : 229
Signal columns   : 40
Anchor concentrations: [0.1, 0.25, 0.5, 1.0, 2.5, 5.0, 7.5, 10.0, 15.0, 25.0, 50.0, 100.0]


---
## 2 · Build Mean Base Curves

Pentru fiecare concentrație din `ANCHOR_CONCENTRATIONS`, calculăm media replicatelor după scăderea baseline-ului. Aceasi functie `compute_mean_baseline_subtracted_signal` din notebookul `data_augumentation.ipynb`, aplicată acum la toate cele 12 concentrații de ancorare.

La 2.5 µM sunt doar 2 replicate.

In [4]:
# build mean base curves for 7.5, 10, 15 µM (avg value for all replicates for each of these concentrations)
def compute_mean_baseline_subtracted_signal(signal_matrix_uA: np.ndarray,
                                            blank_baseline_uA: np.ndarray,
                                            concentration_labels_uM: list[float],
                                            target_concentration_uM: float,
                                            ) -> np.ndarray:
    """For each specific concentration, 
    returns the mean (across replicates) of baseline-subtracted signals. """
    
    # indexes for the target concentration
    column_indices_for_target = [
        i for i, conc in enumerate(concentration_labels_uM)
        if conc == target_concentration_uM
    ]
    
    # subtract baseline to keep only the reaction signal
    baseline_subtracted_replicates = [
        signal_matrix_uA[:, i] - blank_baseline_uA
        for i in column_indices_for_target
    ]
    return np.mean(baseline_subtracted_replicates, axis=0)

base_curves = {
    c: compute_mean_baseline_subtracted_signal(
        raw_signal_matrix_uA, blank_baseline_current_uA, CONCENTRATIONS, c)
    for c in ANCHOR_CONCENTRATIONS
}

print(f"{'Anchor (µM)':>12}  {'Ip (µA)':>10}  {'Ep (V)':>10}  {'Replicates':>10}")
print('-' * 50)
for c in ANCHOR_CONCENTRATIONS:
    n = sum(1 for x in CONCENTRATIONS if x == c)
    pk = Peak(potential_grid_V, savgol_filter(base_curves[c], 5, 3))
    print(f'{c:>12}  {pk.Ip:>10.4f}  {pk.Ep:>10.4f}  {n:>10}')

 Anchor (µM)     Ip (µA)      Ep (V)  Replicates
--------------------------------------------------
         0.1      0.0176     -0.3487           5
        0.25      0.0656     -0.3535           3
         0.5      0.1042     -0.3777           6
         1.0      0.2683     -0.3970           3
         2.5      0.7622     -0.3970           2
         5.0      1.6824     -0.3970           3
         7.5      2.7481     -0.3970           3
        10.0      4.0259     -0.3922           3
        15.0      6.1034     -0.3873           3
        25.0     11.2659     -0.3873           3
        50.0     22.7570     -0.3825           3
       100.0     34.2401     -0.3825           3


---
## 3 · PCHIP Ip–Concentration Spline (Spline PCHIP pentru Ip în funcție de concentrație)

În locul interpolarii liniare simple (din `data_augumentation.ipynb`), pentru a calcula alpha (ponderea de mixare între două curbe), folosesc un spline PCHIP (Piecewise Cubic Hermite Interpolating Polynomial) care:
1. Respectă monotonia, adica nu overshoot între punctele de date (important fizic, Ip trebuie să creasca deodata cu concentratia c)
2. Captează non-linearitatea din zona de saturație (25–100 µM)

La 7.5–15 µM, relația Ip–concentrație e aproape perfect liniară (R² > 0.999), deci interpolarea liniară era suficientă. La intervalul complet, slope-ul scade dramatic după 25 µM din cauza saturației senzorului 

(graficul de mai jos compara PCHIP cu interpolarea liniară simplă (piecewise-linear))


Reference PCHIP: Fritsch & Carlson (1980), *SIAM J. Numer. Anal.* 17(2):238-246.


In [5]:
# Smoothed Ip at each anchor
anchor_peak_currents  = {c: Peak(potential_grid_V, savgol_filter(base_curves[c], 5, 3)).Ip
             for c in ANCHOR_CONCENTRATIONS}

concentration_array = np.array(ANCHOR_CONCENTRATIONS)
peak_current_array = np.array([anchor_peak_currents[c] for c in ANCHOR_CONCENTRATIONS])

# Monotone cubic interpolant (preserves physical monotonicity, no overshoot)
pchip_Ip = PchipInterpolator(concentration_array, peak_current_array)

concentration_fine_grid = np.linspace(concentration_array.min(), concentration_array.max(), 500)


# ----GRAPH----
fig_pchip = go.Figure()
fig_pchip.add_trace(go.Scatter(
    x=concentration_array, y=peak_current_array, mode='markers', name='Empirical anchor',
    marker=dict(color='crimson', size=12, symbol='star',
                line=dict(color='black', width=1))))
fig_pchip.add_trace(go.Scatter(
    x=concentration_fine_grid, y=pchip_Ip(concentration_fine_grid), mode='lines', name='PCHIP spline',
    line=dict(color='steelblue', width=2.6)))
fig_pchip.add_trace(go.Scatter(
    x=concentration_fine_grid, y=np.interp(concentration_fine_grid, concentration_array, peak_current_array), mode='lines',
    name='Piecewise-linear (naive)', line=dict(color='darkorange', width=1.8, dash='dash')))
fig_pchip.update_xaxes(type='log')
apply_default_plotly_layout(fig_pchip, 'Ip vs concentration (PCHIP vs piecewise-linear)',
                             xaxis_title='Concentration (µM, log)', yaxis_title='Ip (µA)')
fig_pchip.show()

# Quantify saturation: slope ratio
low_concentration_data   = pd.DataFrame({'c': concentration_array[concentration_array <= 25], 'Ip': peak_current_array[concentration_array <= 25]})
high_concentration_data = pd.DataFrame({'c': concentration_array[concentration_array >= 25], 'Ip': peak_current_array[concentration_array >= 25]})
slope_low, intercept_low, r_value_low, *_ = linregress(low_concentration_data.c,  low_concentration_data.Ip)
slope_high, intercept_high, r_value_high, *_ = linregress(high_concentration_data.c, high_concentration_data.Ip)
print(f'Linear slope (≤25 µM) : {slope_low:.4f} µA/µM   R²={r_value_low**2:.4f}')
print(f'Saturation slope (≥25) : {slope_high:.4f} µA/µM   R²={r_value_high**2:.4f}')
print(f'Slope ratio low/high   : {slope_low/slope_high:.2f}×  (headline non-linearity)')

Linear slope (≤25 µM) : 0.4459 µA/µM   R²=0.9949
Saturation slope (≥25) : 0.2954 µA/µM   R²=0.9642
Slope ratio low/high   : 1.51×  (headline non-linearity)


---
## 4 · Empirical Noise Model (Long & Winefordner)

Initial, am incercat Lorentzian pentru sigma(c), o formula care amplifica zgomotul la concentratii mici

$$\sigma_{\text{Lor}}(c) = \sigma_0 \cdot \left(1 + \frac{B-1}{1+(c/c_{\text{knee}})^2}\right)$$

Problema:
Cu boost B=3 și c_knee=2.5 µM, rezulta $\sigma(0.1) \approx 3\,\sigma_0 = 3$ nA. Sigma real e in general doar 0.3–0.7 nA. Asta inseamna ca semnalele sintetice sunt generate cu zgomot de 4-10x mai mare decat ce ar fi in realitate, deci peak ul e ingropat in zgomot si semnalul nu mai ajuta.

Fix:
Am ajustat un model fizic, luand in considerare variance ul de la replicatele reale. 

Long & Winefordner (1983) arată că pentru instrumente analitice, variance-ul zgomotului urmează un model combinat absolut-relativ:

$$\sigma(c)^2 = \sigma_{\text{abs}}^2 + (\text{RSD} \cdot I_p(c))^2$$

Unde:
- $\sigma_{\text{abs}}$ = zgomotul de instrument (floor absolut, independent de semnal)
- $\text{RSD}$ = deviația standard relativă (proporțional cu amplitudinea semnalului)

Asta se ajusteaza printr-o regresie liniara simpla a $\sigma^2$ vs $I_p^2$ pe toate replicatele.

**SNR floor:** Am adaugat ulterior o constrangere pentru sigma, sa nu poata depăși $I_p(c) / 3$, adică SNR minim = 3. Fără asta, la concentrații foarte mici unde $I_p$ e mic, chiar și sigma L&W ar putea ingropa peak-ul.

Referință: Long & Winefordner (1983) *Anal. Chem.* 55(7):712A–724A.

In [6]:
def fit_long_winefordner_noise_model(signal_matrix, baseline, concentrations,
                                      anchor_concs, potential_grid):
    """
    Fit σ² = σ_abs² + (RSD·Ip)² by linear regression of σ² vs Ip².
    Returns (sigma_abs_uA, rsd_relative, r_squared).
    """
    means, stds = [], []
    for anchor_conc in anchor_concs:
        replicate_indices = [i for i, conc in enumerate(concentrations) if conc == anchor_conc]
        if len(replicate_indices) < 2:
            continue
        peak_current_replicates = []
        for idx in replicate_indices:
            I = signal_matrix[:, idx] - baseline
            pk = Peak(potential_grid, savgol_filter(I, 5, 3))
            peak_current_replicates.append(pk.Ip)
        means.append(np.mean(peak_current_replicates))
        stds.append(np.std(peak_current_replicates, ddof=1))
    means, stds = np.array(means), np.array(stds)
    slope, intercept, r, *_ = linregress(means**2, stds**2)
    sigma_abs = np.sqrt(max(intercept, 0.0))
    rsd       = np.sqrt(max(slope,     0.0))
    return sigma_abs, rsd, r**2


SIGMA_ABS_uA, RSD_RELATIVE, NOISE_FIT_R2 = fit_long_winefordner_noise_model(
    raw_signal_matrix_uA, blank_baseline_current_uA,
    CONCENTRATIONS, ANCHOR_CONCENTRATIONS, potential_grid_V,
)

print(f'Long & Winefordner noise fit:')
print(f'  σ_abs = {SIGMA_ABS_uA*1e3:.4f} nA')
print(f'  RSD   = {RSD_RELATIVE*100:.3f} %')
print(f'  R²    = {NOISE_FIT_R2:.4f}')

# Hard SNR floor: synthetic signals must have peak-SNR ≥ SNR_FLOOR
SNR_FLOOR = 3.0  # minimum acceptable peak SNR

def lw_noise_sigma(c: float) -> float:
    """
    Long & Winefordner noise σ at concentration c, with SNR floor.

    The SNR floor prevents σ from exceeding Ip(c) / SNR_FLOOR,
    ensuring the augmented signal has a detectable peak.
    """
    Ip_c = float(pchip_Ip(c))
    sigma = np.sqrt(SIGMA_ABS_uA**2 + (RSD_RELATIVE * Ip_c)**2)
    sigma_max = Ip_c / SNR_FLOOR  # hard SNR ceiling on noise
    return float(min(sigma, sigma_max))

# ---- GRAPH ----
# Visualise L&W σ across the full range
concentration_vis_array = np.logspace(np.log10(0.1), np.log10(100), 300)
sigma_lw  = np.array([lw_noise_sigma(c) for c in concentration_vis_array]) * 1e3

fig_noise = go.Figure()
fig_noise.add_trace(go.Scatter(
    x=concentration_vis_array, y=sigma_lw, mode='lines', name='L&W σ (empirical, with SNR floor)',
    line=dict(color='steelblue', width=2.5)))
fig_noise.add_hline(
    y=SIGMA_ABS_uA*1e3, line=dict(color='gray', dash='dot'),
    annotation_text=f'σ_abs = {SIGMA_ABS_uA*1e3:.2f} nA')
fig_noise.update_xaxes(type='log')
apply_default_plotly_layout(
    fig_noise, 'Long & Winefordner σ(c) with SNR floor',
    xaxis_title='Concentration (µM, log)', yaxis_title='σ (nA)')
fig_noise.show()

# Per-anchor SNR table
print(f"\n{'Anchor (µM)':>12}  {'Ip (µA)':>8}  {'σ_LW (nA)':>10}  {'SNR':>6}")
print('-' * 44)
for c in ANCHOR_CONCENTRATIONS:
    Ip = float(pchip_Ip(c))
    s  = lw_noise_sigma(c) * 1e3
    print(f'{c:>12}  {Ip:>8.4f}  {s:>10.3f}  {Ip/(s*1e-3):>6.1f}')

Long & Winefordner noise fit:
  σ_abs = 77.6765 nA
  RSD   = 1.465 %
  R²    = 0.9027



 Anchor (µM)   Ip (µA)   σ_LW (nA)     SNR
--------------------------------------------
         0.1    0.0176       5.853     3.0
        0.25    0.0656      21.856     3.0
         0.5    0.1042      34.739     3.0
         1.0    0.2683      77.776     3.4
         2.5    0.7622      78.475     9.7
         5.0    1.6824      81.493    20.6
         7.5    2.7481      87.488    31.4
        10.0    4.0259      97.528    41.3
        15.0    6.1034     118.437    51.5
        25.0   11.2659     182.399    61.8
        50.0   22.7570     342.293    66.5
       100.0   34.2401     507.555    67.5


---
## 5 · Augmentation Functions

Acum ca exista PCHIP si modelul L&W de zgomot, se poate construi pipeline ul complet de augumentare. Sunt 4 ingrediente principale, fiecare cu un strat diferit de variabilitate:

### 5.1 Interpolare PCHIP-corectată (`get_base_pair` and PCHIP-corrected interpolation)

`get_base_pair` găsește cele două curbe de ancorare care încadrează concentrația tinta. `interpolate_base_curve` calculează alpha-ul de mixare, dar în loc să-l calculeze din concentrație direct (alpha linear în c), îl calculează din Ip-ul PCHIP:

$$\alpha_{\text{PCHIP}} = \frac{\hat{I}_p(c) - \hat{I}_p(c_{\text{low}})}{\hat{I}_p(c_{\text{high}}) - \hat{I}_p(c_{\text{low}})}$$

Asta ajuta in special in zona de saturatie: dacă Ip crește mai lent decat liniar cu c, un alpha calculat din c ar supra-estima Ip-ul sintetic. Alpha din Ip-ul PCHIP reflectă cum se comportă senzorul, nu cum am vrea noi să se comporte.


### 5.2 Zgomot Gaussian pozitional (`apply_position_dependent_noise`)

In loc de zgomot uniform, aplic σ_LW(c) în fereastra de peak și doar σ_abs (floor-ul absolut) în afara ei. Asta reflectă comportamentul real al instrumentului: în zona baseline, zgomotul e dominat de electronic noise constant, nu de variabilitatea semnalului.

### 5.3 Distorsiune de baseline scalată cu concentrația (`apply_polynomial_baseline_distortion`)

Un bump polinomial smooth adăugat la semnal, care simulează background-ul mediului de cultură care pate sa varieze de la o măsurătoare la alta.

**Fix** Amplitudinea era constant 0.05 µA. La 0.1 µM unde Ip e ~0.03–0.05 µA, un bump de 0.05 µA e mai mare decât semnalul în sine, ceea ce e fizic imposibil. Acum amplitudinea scalează cu concentrația:

$$A(c) = A_{\max} \cdot \tanh\!\left(\frac{c}{c_{\text{ref}}}\right)$$


Cu $c_{\text{ref}} = 5$ µM: $A(0.1) \approx 0.02 A_{\max}$, $A(100) \approx A_{\max}$.


Forma polinomului ($A(E_{\text{norm}}^2 - E_{\text{norm}}^4)$) e simetrică față de centrul ferestrei de potențial și zero la margini, standard form for slow-varying electrochemical background drift (Bard & Faulkner 2001, §7.3).

### 5.4 Drift de potențial (`apply_horizontal_potential_drift`)


O deplasare orizontală $\Delta E \sim \mathcal{N}(0, \sigma_E^2)$, $\sigma_E \in [0.005, 0.010]$ V. Modeleaza conditioning drift al electrodului, potențialul de peak nu e exact același de la o măsurătoare la alta. (Osteryoung & Osteryoung 1985, *Anal. Chem.* 57(1):101A–110A).



In [7]:
# Constants
BASELINE_VARIATION_MAX_AMPLITUDE_uA = 0.05   # µA - full amplitude, applied only at high c
BASELINE_SCALE_C_REF_uM             = 5.0    # µM - tanh half-saturation concentration
POTENTIAL_DRIFT_SIGMA_LOW_V         = 0.005   # V
POTENTIAL_DRIFT_SIGMA_HIGH_V        = 0.010   # V


# Segment lookup
def get_base_pair(c_target: float):
    """Return (c_low, I_low, c_high, I_high) bracketing c_target."""
    c_target = float(np.clip(c_target, ANCHOR_CONCENTRATIONS[0], ANCHOR_CONCENTRATIONS[-1]))
    for i in range(len(ANCHOR_CONCENTRATIONS) - 1):
        c_lo, c_hi = ANCHOR_CONCENTRATIONS[i], ANCHOR_CONCENTRATIONS[i + 1]
        if c_lo <= c_target <= c_hi:
            return c_lo, base_curves[c_lo], c_hi, base_curves[c_hi]
    return (ANCHOR_CONCENTRATIONS[-2], base_curves[ANCHOR_CONCENTRATIONS[-2]],
            ANCHOR_CONCENTRATIONS[-1], base_curves[ANCHOR_CONCENTRATIONS[-1]])


# PCHIP-corrected interpolation
def interpolate_base_curve(c_target, c_low, I_low, c_high, I_high, use_pchip=True):
    """
    Blend two anchor curves weighted by PCHIP-corrected alpha.
    """
    if c_high == c_low:
        return I_low.copy()
    if use_pchip:
        Ip_low  = float(pchip_Ip(c_low))
        Ip_high  = float(pchip_Ip(c_high))
        Ip_target   = float(pchip_Ip(c_target))
        denom  = Ip_high - Ip_low
        alpha  = (Ip_target - Ip_low) / denom if abs(denom) > 1e-9 else (c_target - c_low) / (c_high - c_low)
    else:
        # linear interpolation
        alpha = (c_target - c_low) / (c_high - c_low)

    # keep alpha in [0, 1] to avoid extrapolation
    alpha = float(np.clip(alpha, 0.0, 1.0))
    # blend the two anchor curves
    return (1.0 - alpha) * I_low + alpha * I_high


# Gaussian instrument noise (Long & Winefordner)
def apply_gaussian_instrument_noise(signal, noise_sigma_uA):
    """
    Add Gaussian noise σ to every point of the signal.

    σ is supplied by lw_noise_sigma(c), which implements the combined
    absolute-relative model of Long & Winefordner (1983) Anal. Chem. 55(7):712A.
    """
    return signal + np.random.normal(0.0, noise_sigma_uA, signal.shape)

# Noise only in peak window
def apply_position_dependent_noise(signal, potential_grid, c_target,
                                    sigma_abs_uA,
                                    E_peak_start=-0.55, E_peak_end=-0.25):
    """
    Position-dependent Gaussian noise model.

    Applies:
      - σ_LW(c) = sqrt(σ_abs² + (RSD·Ip(c))²)  inside peak window [E_peak_start, E_peak_end]
      - σ_abs                                  outside peak window (baseline region)

    This matches real instrument noise characteristics where off-peak baseline
    noise is dominated by the absolute (instrument-floor) term only.

    Args:
        signal         : 1D current array (µA)
        potential_grid : 1D potential array (V), same length as signal
        c_target       : concentration (µM) used to compute σ_LW
        sigma_abs_uA   : absolute noise floor (µA) from L&W fit
        E_peak_start   : left boundary of peak window (V)
        E_peak_end     : right boundary of peak window (V)

    Returns:
        1D array : noisy signal
    """
    # Full L&W sigma at this concentration
    sigma_peak = lw_noise_sigma(c_target)   # from augmentation notebook

    peak_mask = (potential_grid >= E_peak_start) & (potential_grid <= E_peak_end)
    noise     = np.zeros_like(signal)

    noise[peak_mask]  = np.random.normal(0.0, sigma_peak,  peak_mask.sum())
    noise[~peak_mask] = np.random.normal(0.0, sigma_abs_uA, (~peak_mask).sum())

    return signal + noise


# Concentration-scaled polynomial baseline distortion
def apply_polynomial_baseline_distortion(signal, potential_grid,
                                          c_target,
                                          amp_max_uA=BASELINE_VARIATION_MAX_AMPLITUDE_uA,
                                          c_ref_uM=BASELINE_SCALE_C_REF_uM):
    """
    Add a smooth polynomial baseline bump, amplitude scales with concentration.

    A(c) = amp_max * tanh(c / c_ref)  ->  near-zero at 0.1 µM, full amplitude above 5 µM.

    Shape: amp * (E_norm² - E_norm⁴).  This is an even-symmetric smooth bump
    that is zero at both potential boundaries and peaks near the centre of the
    scan window, consistent with capacitive background drift.
    (Bard & Faulkner 2001, Electrochemical Methods, §7.3).
    """
    # Concentration-dependent amplitude scaling (tanh), 
    # 0 if c_target << c_ref, full amp if c_target >> c_ref
    scale = float(np.tanh(c_target / c_ref_uM))
    amp   = np.random.uniform(-amp_max_uA, +amp_max_uA) * scale
    E_norm = ((potential_grid - potential_grid.mean())
              / (potential_grid.max() - potential_grid.min()))
    distortion = amp * (E_norm**2 - E_norm**4)
    return signal + distortion


# Horizontal potential drift
def apply_horizontal_potential_drift(signal, potential_grid,
                                      sigma_low=POTENTIAL_DRIFT_SIGMA_LOW_V,
                                      sigma_high=POTENTIAL_DRIFT_SIGMA_HIGH_V):
    """
    Shift signal by ΔE ~ N(0, σ²) with σ ~ U(sigma_low, sigma_high).

    Models electrode conditioning drift.
    Reference: Osteryoung & Osteryoung (1985) Anal. Chem. 57(1):101A-110A.
    """
    sigma   = np.random.uniform(sigma_low, sigma_high)
    delta_E = np.random.normal(0.0, sigma)
    # Interpolate the signal to the new potential grid (shifted by delta_E)
    return np.interp(potential_grid, potential_grid + delta_E, signal,
                     left=signal[0], right=signal[-1])


# Full single-signal generation pipeline
def generate_synthetic_signal(c_target, use_pchip=True, use_lw_noise=True,
                               noise_sigma_const=None,
                               baseline_amp_max=BASELINE_VARIATION_MAX_AMPLITUDE_uA,
                               enable_baseline=True, enable_drift=True):
    """
    Full augmentation pipeline for a single concentration.

    Steps: PCHIP interpolation-> L&W noise -> concentration-scaled baseline-> drift.
    """
    c_lo, I_lo, c_hi, I_hi = get_base_pair(c_target)
    I = interpolate_base_curve(c_target, c_lo, I_lo, c_hi, I_hi, use_pchip=use_pchip)

    sigma = lw_noise_sigma(c_target) if use_lw_noise else (noise_sigma_const or 0.001)
    if sigma > 0:
        I = apply_position_dependent_noise(I, potential_grid_V, c_target, sigma)

    if enable_baseline:
        I = apply_polynomial_baseline_distortion(
            I, potential_grid_V, c_target,
            amp_max_uA=baseline_amp_max)

    if enable_drift:
        I = apply_horizontal_potential_drift(I, potential_grid_V)

    return I


print('Augmentation helpers ready.')

Augmentation helpers ready.


---
## 6 · Stratified Sampling Strategy

Avem metoda de generare a semnalului sintetic la o concentratie anume, acum trebuie sa decid cate semnale generam si din ce intervale de concentratie (mai multe cu concentratie redusa, mai putine cu concentratie mare, etc).

Cu un sampling uniform din intervalul [0.1, 100], segmentul 50 - 100 µM este 50% din spatiul de date, segmentul 0.1 - 0.25 µM are doar 0.15 unitati, 0.15% din spatiul total, deci sansele sunt mici sa fie generate date in intervalul acesta, ceea ce ar duce ca un model sa devina expert in a recunoaste semnale mari si orb la semnalele mici. 

**Solutie: sampling log-uniform cu boost pentru coada inferioara.**

Fiecare segment [c_lo, c_hi] primește o greutate proporțională cu lățimea sa pe scală logaritmică:

$$w_k = \ln(c_{k+1}/c_k)$$

Asta se asigura ca sunt acoperite toate intervalele si sunt tratate egal, un segment [0.1, 0.25] primește la fel de multă atenție ca [10, 25] sau [25, 100], chiar dacă primul e de 250× mai ingust pe scala liniara.

**Boost pentru coada joasa (sub 2.5 µM):** 

Segmentele cu concentrație medie sub `LOW_CONC_THRESHOLD = 2.5 µM` primesc cantitate multiplicata cu `LOW_CONC_BOOST`. Asta e similar cu SMOTE (Chawla et al. 2002), adica oversampling pentru clasa minoritara. Fortez generatorul sa dubleze / tripleze cantitatea de date sintetice in zona unde senzorul se "chinuie" cel mai mult (peak urile la concentratie mica nu au forma asa de bine definita ca cele la concentratie mai mare, sunt foarte aproape de background noise)

**Fix implementat:** Boost-ul a fost redus de la 3× la 2×. Cu modelul Lorentzian de zgomot (vechi), boost=3 la 0.1–0.5 µM genera semnale cu peak îngropat, adica modelul primea date care nu aveau un semnal clar și deci nu ajuta la invatare. Cu fix-ul L&W, zgomotul e realist și boost=2 e suficient (dublare).

Bergstra & Bengio (2012) justifică log-uniform sampling ca metoda pentru hyperparameter search (aceeași logică se aplică și pentru sampling-ul concentrațiilor pe o scala care acopera 3 ordine de marime).

In [8]:
# Tunable allocation parameters 
LOW_CONC_THRESHOLD = 2.5  # µM, segments below this get the boost
LOW_CONC_BOOST     = 2.0  # REDUCED from 3.0 (mild oversampling of the low-conc tail)
N_TOTAL_SYNTHETIC  = 300  # total synthetic signals to be generated


def compute_log_allocation(anchor_concs, n_total, low_threshold, boost):
    """
    Log-uniform segment allocation with optional low-concentration boost.

    w_k = log(c_high / c_low): equal coverage per log-decade.
    Segments whose midpoint < low_threshold receive w_k *= boost.
    """
    weights = []
    for i in range(len(anchor_concs) - 1):
        c_low, c_high = anchor_concs[i], anchor_concs[i + 1]
        # "importanta" de baza a segmentului
        # log asigura o acoperire egala pe decada logaritmica
        weight = np.log(c_high / c_low)
        # apply boost if the midpoint of the segment is below the low_threshold
        if (c_low + c_high) / 2.0 < low_threshold:
            weight *= boost
        weights.append(weight)
    
    # normalize weights to sum to 1, then compute counts
    weights  = np.array(weights, dtype=float)
    weights /= weights.sum()
    # how many signals to generate for each segment (3.33 becomes 3)
    counts   = np.round(weights * n_total).astype(int)
    # idenitfy rounding errors (299 or 301 instead of 300)
    diff     = n_total - counts.sum()
    if diff > 0:
        #add a new signal to the segment with the highest weight (most important)
        counts[np.argmax(weights)] += diff
    elif diff < 0:
        # remove a signal from the segment with the lowest weight (least important)
        counts[np.argmin(weights[counts > 0])] -= abs(diff)
    return counts

# used for comparison with log allocation
def compute_uniform_allocation(anchor_concs, n_total):
    """Flat equal allocation across all segments."""
    n_seg  = len(anchor_concs) - 1
    counts = np.full(n_seg, n_total // n_seg, dtype=int)
    counts[:n_total - counts.sum()] += 1
    return counts


segment_counts = compute_log_allocation(
    ANCHOR_CONCENTRATIONS, N_TOTAL_SYNTHETIC, LOW_CONC_THRESHOLD, LOW_CONC_BOOST)

# Print segment allocation summary
print(f"{'Segment':>22}  {'Count':>7}  {'Share %':>8}")
print('-' * 45)
for i, cnt in enumerate(segment_counts):
    seg = f"{ANCHOR_CONCENTRATIONS[i]} – {ANCHOR_CONCENTRATIONS[i+1]} µM"
    flag = '  -- boosted' if ANCHOR_CONCENTRATIONS[i] < LOW_CONC_THRESHOLD else ''
    print(f"{seg:>22}  {cnt:>7}  {cnt/N_TOTAL_SYNTHETIC*100:>7.1f}%{flag}")
print(f"{'TOTAL':>22}  {segment_counts.sum():>7}")

               Segment    Count   Share %
---------------------------------------------
         0.1 – 0.25 µM       54     18.0%  -- boosted
         0.25 – 0.5 µM       41     13.7%  -- boosted
          0.5 – 1.0 µM       41     13.7%  -- boosted
          1.0 – 2.5 µM       54     18.0%  -- boosted
          2.5 – 5.0 µM       21      7.0%
          5.0 – 7.5 µM       12      4.0%
         7.5 – 10.0 µM        8      2.7%
        10.0 – 15.0 µM       12      4.0%
        15.0 – 25.0 µM       15      5.0%
        25.0 – 50.0 µM       21      7.0%
       50.0 – 100.0 µM       21      7.0%
                 TOTAL      300


---
## 7 · Generate the Full Synthetic Dataset

`generate_stratified_dataset` combina toate piesele: pentru fiecare segment, extrage `n_seg` concentratii log-uniform, apeleaza `generate_synthetic_signal` pentru fiecare, si returneaza arrays sortate de concentrații și semnale.

**Important:** Concentrațiile din interiorul unui segment sunt  **log-uniform**: `c ~ exp(U(ln c_lo, ln c_hi))`. Asta e diferit față de `np.random.uniform(c_lo, c_hi)` care risca sa supra reprezinte zona apropiata de c_high pe scala log (de ex. mai multe semnale la 8–10 µM decât la 1–2.5 µM în segmentul [1, 10]).

Reference for log-uniform sampling: Bergstra & Bengio (2012) JMLR 13:281-305.

In [9]:
def generate_stratified_dataset(
        anchor_concs, segment_counts,
        use_pchip=True,
        use_lw_noise=True,
        noise_sigma_const_uA=0.001,
        baseline_amp_uA=BASELINE_VARIATION_MAX_AMPLITUDE_uA,
        enable_baseline=True,
        enable_drift=True,
        rng_seed=GLOBAL_RANDOM_SEED):
    """
    Generate synthetic (concentration, signal) pairs with log-uniform within-segment
    concentration sampling.

    Targets are drawn log-uniformly: c ~ exp(U(ln c_lo, ln c_hi)).
    This ensures equal density per log-decade rather than biasing towards c_hi.

    """
    np.random.seed(rng_seed)
    aug_c, aug_I = [], []
    for i, n_seg in enumerate(segment_counts):
        c_low, c_high = anchor_concs[i], anchor_concs[i + 1]
        # Log-uniform sampling within segment
        log_targets = np.random.uniform(np.log(c_low), np.log(c_high), size=n_seg)
        concentration_targets = np.exp(log_targets)
        for c_t in concentration_targets:
            I = generate_synthetic_signal(
                c_t,
                use_pchip=use_pchip,
                use_lw_noise=use_lw_noise,
                noise_sigma_const=noise_sigma_const_uA,
                baseline_amp_max=baseline_amp_uA,
                enable_baseline=enable_baseline,
                enable_drift=enable_drift,
            )
            aug_c.append(c_t)
            aug_I.append(I)
    order = np.argsort(aug_c)
    return np.array(aug_c)[order], [aug_I[k] for k in order]


# DEFAULT
aug_concentrations, aug_signals_I = generate_stratified_dataset(
    ANCHOR_CONCENTRATIONS, segment_counts)

print(f'Generated {len(aug_signals_I)} synthetic signals')
print(f'Concentration range: {aug_concentrations.min():.3f} – {aug_concentrations.max():.3f} µM')

# Per-segment count check
bins = np.array(ANCHOR_CONCENTRATIONS)
hist, _ = np.histogram(aug_concentrations, bins=bins)
print(f"\n{'Segment':>22}  {'Actual':>8}  {'Planned':>8}")
print('-' * 42)
for i, (cnt, planned) in enumerate(zip(hist, segment_counts)):
    print(f"{f'{ANCHOR_CONCENTRATIONS[i]}–{ANCHOR_CONCENTRATIONS[i+1]} µM':>22}"
          f"  {cnt:>8}  {planned:>8}")

Generated 300 synthetic signals
Concentration range: 0.102 – 98.457 µM

               Segment    Actual   Planned
------------------------------------------
           0.1–0.25 µM        54        54
           0.25–0.5 µM        41        41
            0.5–1.0 µM        41        41
            1.0–2.5 µM        54        54
            2.5–5.0 µM        21        21
            5.0–7.5 µM        12        12
           7.5–10.0 µM         8         8
          10.0–15.0 µM        12        12
          15.0–25.0 µM        15        15
          25.0–50.0 µM        21        21
         50.0–100.0 µM        21        21


---
## 8 · Inspect the Synthetic Dataset

O verificare initiala (cea mai complexa si cuprinzatoare este in validation gate)

Sunt verificate in 2 moduri:
1. **Ip vs concentrație**: Ip-ul extras din fiecare semnal sintetic trebuie sa urmărească  spline-ul PCHIP (linia portocalie). Dacă e mult deviat, avem o problemă cu interpolarea sau zgomotul.

2. **Overlay real vs sintetic**: vizual, semnalele sintetice trebuie să "arate" ca cele reale: același peak, variabilitate similara.

Verificarea de mai jos este pentru debug, validarea se face in notebook ul validation gate.


In [10]:
# Measure Ip from every synthetic curve
Ip_synth = []
for I_s in aug_signals_I:
    try:
        pk = Peak(potential_grid_V, savgol_filter(I_s, 5, 3))
        Ip_synth.append(pk.Ip)
    except Exception:
        Ip_synth.append(np.nan)
Ip_synth = np.array(Ip_synth)
valid = ~np.isnan(Ip_synth)


# ---- GRAPH ----
fig_ip = go.Figure()
fig_ip.add_trace(go.Scatter(
    x=aug_concentrations[valid], y=Ip_synth[valid],
    mode='markers', name='Synthetic Ip',
    marker=dict(color='steelblue', size=5, opacity=0.4)))
fig_ip.add_trace(go.Scatter(
    x=concentration_fine_grid, y=pchip_Ip(concentration_fine_grid),
    mode='lines', name='PCHIP target',
    line=dict(color='darkorange', width=2.4)))
fig_ip.add_trace(go.Scatter(
    x=concentration_array, y=peak_current_array, mode='markers', name='Empirical anchor',
    marker=dict(color='crimson', size=14, symbol='star',
                line=dict(color='black', width=1))))
fig_ip.update_xaxes(type='log')
apply_default_plotly_layout(
    fig_ip, 'Synthetic Ip vs PCHIP target (full range)',
    xaxis_title='Concentration (µM, log)', yaxis_title='Ip (µA)')
fig_ip.show()

# Residual check
print('Per-segment median Ip residual:')
print(f"{'Segment':>22}  {'PCHIP Ip':>10}  {'Median syn':>12}  {'Residual %':>12}")
print('-' * 62)
for i in range(len(ANCHOR_CONCENTRATIONS) - 1):
    c_lo, c_hi = ANCHOR_CONCENTRATIONS[i], ANCHOR_CONCENTRATIONS[i+1]
    mask = (aug_concentrations >= c_lo) & (aug_concentrations < c_hi) & valid
    if mask.sum() == 0:
        continue
    c_mid = np.exp((np.log(c_lo) + np.log(c_hi)) / 2)  # log midpoint
    Ip_t  = float(pchip_Ip(c_mid))
    Ip_m  = float(np.median(Ip_synth[mask]))
    resid = (Ip_m - Ip_t) / Ip_t * 100
    print(f"{f'{c_lo}–{c_hi} µM':>22}  {Ip_t:>10.4f}  {Ip_m:>12.4f}  {resid:>+11.2f}%")

Per-segment median Ip residual:
               Segment    PCHIP Ip    Median syn    Residual %
--------------------------------------------------------------
           0.1–0.25 µM      0.0390        0.0444       +13.99%
           0.25–0.5 µM      0.0825        0.0922       +11.78%
            0.5–1.0 µM      0.1632        0.1974       +20.98%
            1.0–2.5 µM      0.4572        0.5446       +19.12%
            2.5–5.0 µM      1.1287        1.1564        +2.45%
            5.0–7.5 µM      2.1400        2.0342        -4.95%
           7.5–10.0 µM      3.3394        3.6216        +8.45%
          10.0–15.0 µM      4.9706        5.1150        +2.91%
          15.0–25.0 µM      8.2996        8.4995        +2.41%
          25.0–50.0 µM     16.4906       15.6370        -5.18%
         50.0–100.0 µM     28.9123       29.2157        +1.05%


In [11]:
# ---- GRAPH ----
# Real vs synthetic overlay
def plot_real_vs_synthetic(test_concs=(0.5, 10.0, 75.0), n_synth=15):
    fig = make_subplots(rows=1, cols=len(test_concs),
                        subplot_titles=[f'c = {c} µM' for c in test_concs],
                        horizontal_spacing=0.07)
    np.random.seed(7)
    for col, c_t in enumerate(test_concs, 1):
        real_idxs = [i for i, c in enumerate(CONCENTRATIONS) if c == c_t]
        for k, idx in enumerate(real_idxs):
            I_r = raw_signal_matrix_uA[:, idx] - blank_baseline_current_uA
            fig.add_trace(go.Scatter(
                x=potential_grid_V, y=I_r, mode='lines',
                line=dict(color='black', width=1.8), opacity=0.85,
                name='Real' if (k == 0 and col == 1) else None,
                showlegend=(k == 0 and col == 1)), row=1, col=col)
        for k in range(n_synth):
            I_s = generate_synthetic_signal(c_t)
            fig.add_trace(go.Scatter(
                x=potential_grid_V, y=I_s, mode='lines',
                line=dict(color='#17becf', width=0.9), opacity=0.4,
                name='Synthetic' if (k == 0 and col == 1) else None,
                showlegend=(k == 0 and col == 1)), row=1, col=col)
        fig.update_xaxes(range=[-0.65, 0.10], row=1, col=col, showgrid=True)
        fig.update_yaxes(title_text='I (µA)' if col == 1 else None, row=1, col=col)
    fig.update_layout(
        height=500, width=1100, template=PLOTLY_TEMPLATE,
        title=dict(text='Real vs synthetic (low / mid / high)', x=0.5, xanchor='center'),
        legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='right', x=1.0),
        margin=dict(l=60, r=30, t=90, b=60))
    return fig

plot_real_vs_synthetic(test_concs=[0.5, 10.0, 75.0]).show()

---
## 9 · Feature Extraction (Signal + vectorize pipeline)

Aceeași funcție `vectorize_signal_features` din `data_augmentation.ipynb`, cu aceleasi 3 suite: `core`, `extended`, `experimental`.

Pentru datele sintetice, baseline-ul a fost deja scăzut în etapa de construire a anchor curves (Secțiunea 2). Deci cand punem un semnal sintetic în clasa `Signal`, trebuie resetat `Signal.set_common_baseline_I([])`, pentru a nu se mai scadea baseline-ul din nou.

Celula urmatoare construieste DataFrame-urile pentru toate combinațiile de (original / augmentat / combinat) × (suite) și le salveaza drept CSV-uri in folderul `vectorized/`. 

De referinta: `full_combined_experimental.csv`, contine tot.

Celula de la final (raw signals) salveaza arrayurile cu semnalele brute pentru a putea fi analizate in validation gate. 



In [12]:
def vectorize_signal_features(sig_obj, suite='core'):
    vec = []
    if suite in ('core', 'extended', 'experimental'):
        vec += [sig_obj.get_peak_current_value(), sig_obj.get_peak_potential_value(),
                sig_obj.get_peak_auc(),            sig_obj.get_peak_fwhm()]
    if suite in ('extended', 'experimental'):
        vec += [sig_obj.get_pca1_comp(),
                sig_obj.get_first_derivative_max(),
                sig_obj.get_second_derivative_min()]
    if suite == 'experimental':
        vec += [
            sig_obj.get_left_slope(), sig_obj.get_right_slope(),
            sig_obj.get_asymetry(),   sig_obj.get_peak_sharpness(),
            sig_obj.get_peak_compactness(), sig_obj.get_current_variance(),
            sig_obj.get_peak_skewness(),    sig_obj.get_peak_kurtosis(),
            sig_obj.get_tchebichef_curve_moments(), sig_obj.get_mean_peak(),
            sig_obj.get_signal_entropy(), sig_obj.get_spectral_entropy(),
            sig_obj.get_fft_power(),
            sig_obj.get_pca2_comp(), sig_obj.get_pca3_comp(),
            sig_obj.get_wavelet_energy(),
        ]
    return vec


FEATURE_COLUMN_NAMES_BY_SUITE = {
    'core': ['peak_current','peak_potential','peak_AUC','peak_FWHM'],
    'extended': ['peak_current','peak_potential','peak_AUC','peak_FWHM',
                 'pca1_comp','first_derivative_max','second_derivative_min'],
    'experimental': ['peak_current','peak_potential','peak_AUC','peak_FWHM',
                     'pca1_comp','first_derivative_max','second_derivative_min',
                     'left_slope','right_slope','asymetry','peak_sharpness',
                     'peak_compactness','current_variance','peak_skewness',
                     'peak_kurtosis','tchebichef_curve_moments','mean_peak',
                     'signal_entropy','spectral_entropy','fft_power',
                     'pca2_comp','pca3_comp','wavelet_energy'],
}

# Process original signals
Signal.set_common_potential_E(potential_grid_V)
Signal.set_common_baseline_I(blank_baseline_current_uA)

original_signals_with_labels = []
for col_idx in range(raw_signal_matrix_uA.shape[1]):
    try:
        sig = Signal(raw_signal_matrix_uA[:, col_idx])
        original_signals_with_labels.append((sig, CONCENTRATIONS[col_idx]))
    except Exception as e:
        print(f'  [WARN] original signal {col_idx} skipped: {e}')

print(f'Original signals processed: {len(original_signals_with_labels)}')

Original signals processed: 40


In [13]:
def process_synthetic_into_signal_objects(syn_signals, syn_concs):
    """Wrap synthetic current arrays in Signal objects (baseline bypass)."""
    Signal.set_common_baseline_I(np.array([]))  # bypass subtraction
    objs, n_skip = [], 0
    for I, c in zip(syn_signals, syn_concs):
        try:
            objs.append((Signal(I), float(c)))
        except Exception:
            n_skip += 1
    Signal.set_common_baseline_I(blank_baseline_current_uA)  # restore
    return objs, n_skip


def build_feature_dataframe(sig_label_pairs, suite):
    rows, n_skip = [], 0
    for sig, c in sig_label_pairs:
        try:
            rows.append(vectorize_signal_features(sig, suite) + [c])
        except Exception:
            n_skip += 1
    if n_skip > 0:
        print(f'  [{suite}] skipped {n_skip} signals')
    cols = FEATURE_COLUMN_NAMES_BY_SUITE[suite] + ['concentration']
    return pd.DataFrame(rows, columns=cols)


default_syn_objs, default_n_skip = process_synthetic_into_signal_objects(
    aug_signals_I, aug_concentrations)
print(f'Synthetic signals processed: {len(default_syn_objs)}  (skipped: {default_n_skip})')

# Build DataFrames
orig_core = build_feature_dataframe(original_signals_with_labels, 'core')
orig_ext  = build_feature_dataframe(original_signals_with_labels, 'extended')
orig_exp  = build_feature_dataframe(original_signals_with_labels, 'experimental')

aug_core  = build_feature_dataframe(default_syn_objs, 'core')
aug_ext   = build_feature_dataframe(default_syn_objs, 'extended')
aug_exp   = build_feature_dataframe(default_syn_objs, 'experimental')

combined_core = pd.concat([orig_core, aug_core], ignore_index=True)
combined_ext  = pd.concat([orig_ext,  aug_ext],  ignore_index=True)
combined_exp  = pd.concat([orig_exp,  aug_exp],  ignore_index=True)

print(f'Original  - core: {orig_core.shape}   extended: {orig_ext.shape}   experimental: {orig_exp.shape}')
print(f'Augmented - core: {aug_core.shape}   extended: {aug_ext.shape}   experimental: {aug_exp.shape}')
print(f'Combined  - core: {combined_core.shape}  extended: {combined_ext.shape}   experimental: {combined_exp.shape}')

Synthetic signals processed: 300  (skipped: 0)
Original  - core: (40, 5)   extended: (40, 8)   experimental: (40, 24)
Augmented - core: (300, 5)   extended: (300, 8)   experimental: (300, 24)
Combined  - core: (340, 5)  extended: (340, 8)   experimental: (340, 24)


In [14]:
# Persist feature CSVs
paths.VECTORIZED_DIR.mkdir(exist_ok=True)

for name, df in [
    ('full_augmented_core',         aug_core),
    ('full_augmented_extended',     aug_ext),
    ('full_augmented_experimental', aug_exp),
    ('full_combined_core',          combined_core),
    ('full_combined_extended',      combined_ext),
    ('full_combined_experimental',  combined_exp),
]:
    path = paths.VECTORIZED_DIR / f'{name}.csv'
    df.to_csv(path, index=False)
    print(f'  Saved {path.relative_to(paths.ROOT)}  ({df.shape[0]} rows × {df.shape[1]} cols)')

  Saved vectorized\full_augmented_core.csv  (300 rows × 5 cols)
  Saved vectorized\full_augmented_extended.csv  (300 rows × 8 cols)
  Saved vectorized\full_augmented_experimental.csv  (300 rows × 24 cols)
  Saved vectorized\full_combined_core.csv  (340 rows × 5 cols)
  Saved vectorized\full_combined_extended.csv  (340 rows × 8 cols)
  Saved vectorized\full_combined_experimental.csv  (340 rows × 24 cols)


### Save Raw Signals for Validation Gate

Validation gate (`validation_gate.ipynb`) foloseste raw current arrays (`n_signals × n_voltage_points`) pentru testele legate de forma curbei (KS, JSD, MMD², SWD, ACF, Randles-Ševčík, PCA/t-SNE overlays). 
Am salvat 3 fisiere cu semnalele generate in formatul implicit raw:

* **`raw/raw_signals_real.csv`** - Real signals only (40 × `n_pts`)
* **`raw/raw_signals_augmented.csv`** - Augmented only (300 × `n_pts`)
* **`raw/raw_signals_combined.csv`** - Real + augmented (340 × `n_pts`)

---

**Data Structure**
* **Row format:** The first column is `concentration` (µM), and the remaining columns are the current values `I[0..n-1]` (µA).
* **Potential grid:** The potential grid is shared for all signals and is saved separately in `raw_potential_grid.csv`.

In [15]:
paths.RAW_DIR.mkdir(exist_ok=True)
# Potential grid 
pd.DataFrame({'potential_V': potential_grid_V}).to_csv(
    paths.POTENTIAL_GRID_CSV, index=False)
print(f'Saved {paths.POTENTIAL_GRID_CSV.name}  ({len(potential_grid_V)} points)')

# Helper: DataFrame with concentration + raw I columns
def _raw_to_df(I_matrix, concentrations):
    """I_matrix: (n_signals, n_voltage_points) ndarray; concentrations: (n_signals,)"""
    col_names = [f'I_{k}' for k in range(I_matrix.shape[1])]
    df = pd.DataFrame(I_matrix, columns=col_names)
    df.insert(0, 'concentration', concentrations)
    return df

# Real signals 
# Reconstruct from the original_signals_with_labels list used in the
# vectorization step. 
# Each entry is (Signal_object, concentration).
# raw baseline-subtracted current is Signal._I_raw_subtracted
# or raw_signal_matrix_uA[:, col] - blank_baseline_current_uA.
real_I_list, real_c_list = [], []
for col_idx in range(raw_signal_matrix_uA.shape[1]):
    try:
        I_sub = raw_signal_matrix_uA[:, col_idx] - blank_baseline_current_uA
        real_I_list.append(I_sub)
        real_c_list.append(CONCENTRATIONS[col_idx])
    except Exception as e:
        print(f'  [WARN] real signal {col_idx} skipped: {e}')

X_real_raw = np.array(real_I_list) # (n_real, n_pts)
y_real_raw = np.array(real_c_list) # (n_real,)

df_real = _raw_to_df(X_real_raw, y_real_raw)
df_real.to_csv(paths.REAL_SIGNALS_CSV , index=False)
print(f'Saved {paths.REAL_SIGNALS_CSV.relative_to(paths.ROOT)}   ({df_real.shape[0]} rows × {df_real.shape[1]} cols)')

# Augmented signals
# aug_signals_I is a list of 1-D current arrays
# aug_concentrations is the matching concentration array 
# both were produced by generate_stratified_dataset().
X_aug_raw = np.array(aug_signals_I) # (n_aug, n_pts)
y_aug_raw = aug_concentrations # (n_aug,)

df_aug = _raw_to_df(X_aug_raw, y_aug_raw)
df_aug.to_csv(paths.AUGMENTED_SIGNALS_CSV, index=False)
print(f'Saved {paths.AUGMENTED_SIGNALS_CSV.relative_to(paths.ROOT)} ({df_aug.shape[0]} rows × {df_aug.shape[1]} cols)')

# Combined (real + augmented)
X_combined_raw = np.vstack([X_real_raw, X_aug_raw]) # (n_real+n_aug, n_pts)
y_combined_raw = np.concatenate([y_real_raw, y_aug_raw]) # (n_real+n_aug,)

df_combined = _raw_to_df(X_combined_raw, y_combined_raw)
df_combined.to_csv(paths.COMBINED_SIGNALS_CSV, index=False)
print(f'Saved {paths.COMBINED_SIGNALS_CSV.relative_to(paths.ROOT)}   ({df_combined.shape[0]} rows × {df_combined.shape[1]} cols)')

print(f'\nPotential grid : {len(potential_grid_V)} points  '
      f'({potential_grid_V[0]:.4f} V → {potential_grid_V[-1]:.4f} V)')
print(f'Real signals   : {X_real_raw.shape[0]}   '
      f'  conc range {y_real_raw.min():.2f}–{y_real_raw.max():.2f} µM')
print(f'Aug signals    : {X_aug_raw.shape[0]}  '
      f'  conc range {y_aug_raw.min():.3f}–{y_aug_raw.max():.3f} µM')
print(f'Combined       : {X_combined_raw.shape[0]}  '
      f'  conc range {y_combined_raw.min():.3f}–{y_combined_raw.max():.3f} µM')


Saved raw_potential_grid.csv  (229 points)
Saved raw\raw_signals_real.csv   (40 rows × 230 cols)
Saved raw\raw_signals_augmented.csv (300 rows × 230 cols)
Saved raw\raw_signals_combined.csv   (340 rows × 230 cols)

Potential grid : 229 points  (-0.6001 V → 0.5023 V)
Real signals   : 40     conc range 0.10–100.00 µM
Aug signals    : 300    conc range 0.102–98.457 µM
Combined       : 340    conc range 0.100–100.000 µM


---
## 10 · Ablation Study

Next section includes the evaluation of each augumentation strategy. For generating synthetic data, only the sections above are relevant.

Ablation study-ul compara variantele de augmentare pentru a justifica alegerile din `S_REFINED`. Fiecare strategie e identificată printr-un ID (ex. `S2_lw_noise_only`) și un label descriptiv.

**Changes vs the diagnostic notebook / original `data_augumentation.ipynb` :**
- `DecisionTree` removed (overfits 39-sample originals → Protocol A MAE aprox 0, skews all result plots)
- `S8_all_default` replaced by `S_REFINED` (L&W noise + scaled baseline + log-uniform sampling + boost=2)
- Added `S_OLD_ALL` for direct comparison with the old failing strategy
- Added `S_INTERP_PLUS_NOISE` as a minimal but effective baseline

### Evaluation protocols
( Protocolul B era folosit in `data_augumentation.ipynb` strict pentru a justifica nevoia pentru protocolul C )
- **Protocol A**: LOOCV on originals only (reference)
- **Protocol C**: hold out one original, train on rest + all synthetic
- **ΔMAE = MAE_C − MAE_A**: negative = augmentation helps; positive = augmentation hurts

In [43]:
N_TOTAL_SYNTHETIC_ABLATION = 200

ABLATION_MODELS = {
    'Ridge': make_pipeline(
        SimpleImputer(strategy='median'),
        Ridge(alpha=0.1, fit_intercept=True, random_state=42)),
    'RandomForest': RandomForestRegressor(
        n_estimators=500, max_depth=9, max_features=0.75,
        min_samples_split=2, min_samples_leaf=1,
        random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(
        n_estimators=235, max_depth=3, learning_rate=0.0698,
        subsample=0.7682, colsample_bytree=0.9395,
        gamma=0.3391, min_child_weight=3,
        random_state=42, verbosity=0, n_jobs=-1),
}

ABLATION_FEATURE_SUITE = 'experimental'
ABLATION_FEATURE_COLS  = FEATURE_COLUMN_NAMES_BY_SUITE[ABLATION_FEATURE_SUITE]

print(f"Models: {list(ABLATION_MODELS.keys())}")
print(f"Feature suite: {ABLATION_FEATURE_SUITE} ({len(ABLATION_FEATURE_COLS)} features)")

Models: ['Ridge', 'RandomForest', 'XGBoost']
Feature suite: experimental (23 features)


In [44]:
# Parametric generators (reused by ablation loop)

def _sigma_lw_parametric(c, sigma_abs, rsd):
    """L&W σ for parametric ablation sweeps."""
    Ip_c = float(pchip_Ip(c))
    s    = np.sqrt(sigma_abs**2 + (rsd * Ip_c)**2)
    return float(min(s, Ip_c / SNR_FLOOR))


def generate_ablation_dataset(strategy_cfg, n_total=N_TOTAL_SYNTHETIC_ABLATION,
                               rng_seed=GLOBAL_RANDOM_SEED):
    """Dispatch a strategy dict to the dataset generator."""
    if not strategy_cfg.get('use_interpolation', True):
        return np.array([]), []

    boost = strategy_cfg.get('boost', LOW_CONC_BOOST)

    if strategy_cfg.get('stratified', True):
        seg_counts = compute_log_allocation(
            ANCHOR_CONCENTRATIONS, n_total, LOW_CONC_THRESHOLD, boost)
    else:
        seg_counts = compute_uniform_allocation(ANCHOR_CONCENTRATIONS, n_total)

    return generate_stratified_dataset(
        ANCHOR_CONCENTRATIONS, seg_counts,
        use_pchip=strategy_cfg.get('use_pchip', True),
        use_lw_noise=strategy_cfg.get('use_lw_noise', True),
        noise_sigma_const_uA=strategy_cfg.get('noise_sigma_uA', 0.001),
        baseline_amp_uA=strategy_cfg.get('baseline_amp_uA', BASELINE_VARIATION_MAX_AMPLITUDE_uA),
        enable_baseline=strategy_cfg.get('enable_baseline', True),
        enable_drift=strategy_cfg.get('enable_drift', True),
        rng_seed=rng_seed,
    )


ABLATION_STRATEGIES_CONFIG = [
    # REFERENCE
    dict(id='S0_no_aug',
         use_interpolation=False,
         label='No augmentation (Protocol A only)'),

    # INTERPOLATION ONLY
    dict(id='S1_interp_only',
         use_pchip=True, use_lw_noise=False, noise_sigma_uA=0.0,
         enable_baseline=False, enable_drift=False, stratified=True, boost=LOW_CONC_BOOST,
         label='PCHIP interpolation only'),

    # NOISE SIGMA SWEEP (L&W noise, no baseline or drift) 
    dict(id='S2_lw_noise_only',
         use_pchip=True, use_lw_noise=True, noise_sigma_uA=0.001,
         enable_baseline=False, enable_drift=False, stratified=True, boost=LOW_CONC_BOOST,
         label='PCHIP + L&W noise only'),
    dict(id='S3_const_noise_001',
         use_pchip=True, use_lw_noise=False, noise_sigma_uA=0.001,
         enable_baseline=False, enable_drift=False, stratified=True, boost=LOW_CONC_BOOST,
         label='PCHIP + constant σ=0.001 µA'),
    dict(id='S4_const_noise_005',
         use_pchip=True, use_lw_noise=False, noise_sigma_uA=0.005,
         enable_baseline=False, enable_drift=False, stratified=True, boost=LOW_CONC_BOOST,
         label='PCHIP + constant σ=0.005 µA'),

    # SINGLE-INGREDIENT ABLATIONS 
    dict(id='S5_baseline_only',
         use_pchip=True, use_lw_noise=False, noise_sigma_uA=0.0,
         enable_baseline=True, enable_drift=False, stratified=True, boost=LOW_CONC_BOOST,
         label='PCHIP + scaled baseline only'),
    dict(id='S6_drift_only',
         use_pchip=True, use_lw_noise=False, noise_sigma_uA=0.0,
         enable_baseline=False, enable_drift=True, stratified=True, boost=LOW_CONC_BOOST,
         label='PCHIP + potential drift only'),

    # KEY COMPARISON: OLD vs REFINED "ALL" 
    dict(id='S_OLD_ALL',
         use_pchip=True, use_lw_noise=False, noise_sigma_uA=0.001,
         enable_baseline=True, enable_drift=True, stratified=True, boost=3.0,
         # Use constant sigma (not L&W) and old boost=3 to replicate old S8_all_default
         label='OLD all-default (const sigma, flat baseline, boost=3) - REFERENCE'),
    dict(id='S_REFINED',
         use_pchip=True, use_lw_noise=True, noise_sigma_uA=0.001,
         enable_baseline=True, enable_drift=True, stratified=True, boost=LOW_CONC_BOOST,
         label='REFINED (L&W noise + scaled baseline + log sampling + boost=2)  ← RECOMMENDED'),

    # FULL-RANGE-SPECIFIC ABLATIONS
    dict(id='S7_linear_alpha',
         use_pchip=False, use_lw_noise=True, noise_sigma_uA=0.001,
         enable_baseline=True, enable_drift=True, stratified=True, boost=LOW_CONC_BOOST,
         label='REFINED but linear alpha (no PCHIP correction)'),
    dict(id='S8_uniform_sampling',
         use_pchip=True, use_lw_noise=True, noise_sigma_uA=0.001,
         enable_baseline=True, enable_drift=True, stratified=False, boost=LOW_CONC_BOOST,
         label='REFINED but uniform allocation (no oversampling)'),
    dict(id='S9_const_sigma',
         use_pchip=True, use_lw_noise=False, noise_sigma_uA=0.001,
         enable_baseline=True, enable_drift=True, stratified=True, boost=LOW_CONC_BOOST,
         label='REFINED but constant sigma (no L&W)'),
    dict(id='S10_no_low_conc_boost',
         use_pchip=True, use_lw_noise=True, noise_sigma_uA=0.001,
         enable_baseline=True, enable_drift=True, stratified=True,
         label='REFINED but boost=1 (no low-conc oversampling)', boost=1.0),

    # BOOST SWEEP 
    dict(id='S11_boost_1p5',
         use_pchip=True, use_lw_noise=True, noise_sigma_uA=0.001,
         enable_baseline=True, enable_drift=True, stratified=True, boost=1.5,
         label='REFINED boost=1.5'),
    dict(id='S12_boost_3',
         use_pchip=True, use_lw_noise=True, noise_sigma_uA=0.001,
         enable_baseline=True, enable_drift=True, stratified=True, boost=3.0,
         label='REFINED boost=3.0'),
    dict(id='S13_boost_4',
         use_pchip=True, use_lw_noise=True, noise_sigma_uA=0.001,
         enable_baseline=True, enable_drift=True, stratified=True, boost=4.0,
         label='REFINED boost=4.0'),
]

print(f'Configured {len(ABLATION_STRATEGIES_CONFIG)} ablation strategies.')

Configured 16 ablation strategies.


In [45]:
# Evaluation functions

def protocol_A_loocv(feat_df, feat_cols, model_obj):
    X, y = feat_df[feat_cols], feat_df['concentration']
    y_true, y_pred = [], []
    for train_idx, test_idx in LeaveOneOut().split(X):
        m = clone(model_obj)
        m.fit(X.iloc[train_idx], y.iloc[train_idx])
        y_pred.append(m.predict(X.iloc[test_idx])[0])
        y_true.append(y.iloc[test_idx].iloc[0])
    true_concentrations_array, predicted_concentrations_array = np.array(y_true), np.array(y_pred)
    return dict(MAE=float(mean_absolute_error(true_concentrations_array, predicted_concentrations_array)),
                RMSE=float(np.sqrt(mean_squared_error(true_concentrations_array, predicted_concentrations_array))),
                R2=float(r2_score(true_concentrations_array, predicted_concentrations_array)))


def protocol_C_held_out(orig_df, syn_df, feat_cols, model_obj):
    X_o, y_o = orig_df[feat_cols], orig_df['concentration']
    X_s, y_s = syn_df[feat_cols],  syn_df['concentration']
    y_true, y_pred = [], []
    for held in range(len(X_o)):
        kept    = [i for i in range(len(X_o)) if i != held]
        X_train = pd.concat([X_o.iloc[kept], X_s], ignore_index=True)
        y_train = pd.concat([y_o.iloc[kept], y_s], ignore_index=True)
        m = clone(model_obj)
        m.fit(X_train, y_train)
        y_pred.append(m.predict(X_o.iloc[[held]])[0])
        y_true.append(y_o.iloc[held])
    true_concentrations_array, predicted_concentrations_array = np.array(y_true), np.array(y_pred)
    return dict(MAE=float(mean_absolute_error(true_concentrations_array, predicted_concentrations_array)),
                RMSE=float(np.sqrt(mean_squared_error(true_concentrations_array, predicted_concentrations_array))),
                R2=float(r2_score(true_concentrations_array, predicted_concentrations_array)))


print('Evaluation functions ready.')

Evaluation functions ready.


In [46]:
# Run the ablation
ablation_result_rows = []

print('Protocol A baseline (LOOCV, originals only)...')
protocol_A_by_model = {}
for mn, mo in ABLATION_MODELS.items():
    protocol_A_by_model[mn] = protocol_A_loocv(orig_exp, ABLATION_FEATURE_COLS, mo)
    m = protocol_A_by_model[mn]
    print(f'  {mn:14s}  MAE={m["MAE"]:.4f}  RMSE={m["RMSE"]:.4f}  R2={m["R2"]:.4f}')
print()

for cfg in ABLATION_STRATEGIES_CONFIG:
    sid = cfg['id']
    print(f'  {sid:30s}  {cfg["label"][:60]}')

    sc, sI = generate_ablation_dataset(cfg, n_total=N_TOTAL_SYNTHETIC_ABLATION)

    if not cfg.get('use_interpolation', True) or len(sI) == 0:
        for mn in ABLATION_MODELS:
            mA = protocol_A_by_model[mn]
            ablation_result_rows.append(dict(
                Strategy=sid, Label=cfg['label'], Model=mn,
                **{f'ProtA_{k}': round(v, 4) for k, v in mA.items()},
                **{f'ProtC_{k}': round(v, 4) for k, v in mA.items()},
                dMAE=0.0, dR2=0.0))
        continue

    sobjs, _ = process_synthetic_into_signal_objects(sI, sc)
    sdf      = build_feature_dataframe(sobjs, ABLATION_FEATURE_SUITE)

    for mn, mo in ABLATION_MODELS.items():
        mA = protocol_A_by_model[mn]
        mC = protocol_C_held_out(orig_exp, sdf, ABLATION_FEATURE_COLS, mo)
        ablation_result_rows.append(dict(
            Strategy=sid, Label=cfg['label'], Model=mn,
            **{f'ProtA_{k}': round(v, 4) for k, v in mA.items()},
            **{f'ProtC_{k}': round(v, 4) for k, v in mC.items()},
            dMAE=round(mC['MAE'] - mA['MAE'], 4),
            dR2=round(mC['R2']  - mA['R2'],  4)))
6
ablation_df = pd.DataFrame(ablation_result_rows)
ablation_df.to_csv('full_range_ablation_results_refined.csv', index=False)
print(f'\nAblation complete: {len(ablation_df)} rows saved.')

Protocol A baseline (LOOCV, originals only)...
  Ridge           MAE=0.6414  RMSE=1.0574  R2=0.9985
  RandomForest    MAE=1.0646  RMSE=2.2509  R2=0.9933
  XGBoost         MAE=1.8375  RMSE=4.4595  R2=0.9735

  S0_no_aug                       No augmentation (Protocol A only)
  S1_interp_only                  PCHIP interpolation only
  [experimental] skipped 1 signals
  S2_lw_noise_only                PCHIP + L&W noise only
  [experimental] skipped 1 signals
  S3_const_noise_001              PCHIP + constant σ=0.001 µA
  [experimental] skipped 1 signals
  S4_const_noise_005              PCHIP + constant σ=0.005 µA
  [experimental] skipped 1 signals
  S5_baseline_only                PCHIP + scaled baseline only
  [experimental] skipped 2 signals
  S6_drift_only                   PCHIP + potential drift only
  [experimental] skipped 2 signals
  S_OLD_ALL                       OLD all-default (const sigma, flat baseline, boost=3) - REFE
  [experimental] skipped 2 signals
  S_REFINED        

In [47]:
# ---- GRAPH ----
# ΔMAE heatmap (Ridge / RF / XGBoost)
strategy_order = [s['id'] for s in ABLATION_STRATEGIES_CONFIG]
strategy_labels = {s['id']: s['label'][:55] for s in ABLATION_STRATEGIES_CONFIG}

delta_pivot = ablation_df.pivot_table(
    index='Strategy', columns='Model', values='dMAE', aggfunc='first',
).reindex(index=strategy_order)

max_abs = max(abs(delta_pivot.values.min()), abs(delta_pivot.values.max()))

heatmap_fig = go.Figure(data=go.Heatmap(
    z=delta_pivot.values,
    x=delta_pivot.columns.tolist(),
    y=[strategy_labels.get(s, s) for s in delta_pivot.index],
    colorscale='RdBu', zmid=0.0, zmin=-max_abs, zmax=+max_abs,
    text=np.round(delta_pivot.values, 3),
    texttemplate='%{text:+.3f}',
    hovertemplate='Strategy: %{y}<br>Model: %{x}<br>ΔMAE = %{z:+.4f}<extra></extra>',
    colorbar=dict(title='ΔMAE<br>(C−A)'),
))
heatmap_fig.update_layout(
    height=750, width=680,
    template=PLOTLY_TEMPLATE,
    title=dict(text='ΔMAE heatmap (red / negative values = augmentation helps)',
               x=0.5, xanchor='center'),
    margin=dict(l=420, r=30, t=70, b=50),
)
heatmap_fig.show()

# Summary: best strategy per model
print('\nBest strategy per model (lowest Protocol C MAE):')
for mn in ABLATION_MODELS:
    sub = ablation_df[ablation_df['Model'] == mn]
    best = sub.loc[sub['ProtC_MAE'].idxmin()]
    print(f'  {mn:14s}  {best["Strategy"]:30s}  '
          f'ProtC MAE={best["ProtC_MAE"]:.4f}  ΔMAE={best["dMAE"]:+.4f}')


Best strategy per model (lowest Protocol C MAE):
  Ridge           S7_linear_alpha                 ProtC MAE=0.4032  ΔMAE=-0.2382
  RandomForest    S7_linear_alpha                 ProtC MAE=0.3336  ΔMAE=-0.7309
  XGBoost         S13_boost_4                     ProtC MAE=0.2709  ΔMAE=-1.5666


In [48]:
# Protocol C MAE bar chart
model_colors = {'Ridge': '#1f77b4', 'RandomForest': '#2ca02c', 'XGBoost': '#d62728'}

bar_fig = go.Figure()
for mn in ABLATION_MODELS:
    rows = (ablation_df[ablation_df['Model'] == mn]
            .set_index('Strategy').reindex(strategy_order).reset_index())
    bar_fig.add_trace(go.Bar(
        x=rows['Strategy'], y=rows['ProtC_MAE'],
        name=mn, marker_color=model_colors[mn], opacity=0.85,
        hovertemplate=f'{mn}<br>%{{x}}<br>MAE=%{{y:.4f}}<extra></extra>'))

# No-aug reference lines
for mn in ABLATION_MODELS:
    noaug_mae = ablation_df[
        (ablation_df['Strategy'] == 'S0_no_aug') &
        (ablation_df['Model'] == mn)]['ProtC_MAE'].iloc[0]
    bar_fig.add_hline(y=noaug_mae,
                      line=dict(color=model_colors[mn], dash='dot', width=1.0),
                      annotation_text=f'{mn} no-aug', annotation_position='top left',
                      annotation_font_size=9)

bar_fig.update_layout(
    barmode='group', height=560, width=1400, template=PLOTLY_TEMPLATE,
    title=dict(text='Protocol C MAE — Refined ablation study (no DecisionTree)',
               x=0.5, xanchor='center'),
    xaxis_title='Strategy', yaxis_title='Protocol C MAE (µM)',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0),
    margin=dict(l=60, r=30, t=80, b=160))
bar_fig.update_xaxes(tickangle=-35)
bar_fig.show()

---
## 11 · SNR (Signal-to-Noise Ratio) Audit (Edge-Case Validation)

Verifies that the L&W noise model + SNR floor keep all anchor concentrations above the SNR threshold.



In [54]:
N_SNR_TEST = 80
np.random.seed(99)

snr_rows = []
for c in ANCHOR_CONCENTRATIONS:
    Ip_target = float(pchip_Ip(c))
    snrs = []
    for _ in range(N_SNR_TEST):
        I = generate_synthetic_signal(c)
        E_peak_start, E_peak_end = -0.55, -0.25
        flank_mask = (
        ((potential_grid_V >= -0.65) & (potential_grid_V < E_peak_start)) |
        ((potential_grid_V > E_peak_end)  & (potential_grid_V <= -0.15))
        )
        noise_est = np.std(I[flank_mask]) if flank_mask.sum() > 5 else np.std(I[:10])
        try:
            pk  = Peak(potential_grid_V, savgol_filter(I, 5, 3))
            snr = pk.Ip / noise_est if noise_est > 1e-9 else np.inf
        except Exception:
            snr = 0.0
        snrs.append(snr)
    snrs = np.array(snrs)
    snr_rows.append({
        'concentration (µM)': c,
        'Ip_target (µA)':     round(Ip_target, 5),
        'σ_LW (nA)':          round(lw_noise_sigma(c)*1e3, 3),
        'median_SNR':         round(float(np.median(snrs)), 2),
        'min_SNR':            round(float(np.min(snrs)), 2),
        'pct_below_3 (%)':    round(float(np.mean(snrs < 3)*100), 1),
        'pct_below_2 (%)':    round(float(np.mean(snrs < 2)*100), 1),
    })

snr_df = pd.DataFrame(snr_rows)
print('SNR audit (n=80 signals per anchor):')
print(snr_df.to_string(index=False))

bad = snr_df[snr_df['pct_below_3 (%)'] > 20]
if len(bad):
    print(f'\n Anchors with >20% signals below SNR 3: {bad["concentration (µM)"].tolist()}')
    print(f'expected: 0.10 µM, since it is right at the limit of detection')
else:
    print('\n All anchors pass the SNR 3 threshold.')

SNR audit (n=80 signals per anchor):
 concentration (µM)  Ip_target (µA)  σ_LW (nA)  median_SNR  min_SNR  pct_below_3 (%)  pct_below_2 (%)
               0.10         0.01756      5.853        2.27     1.42             97.5             23.8
               0.25         0.06557     21.856        4.08     2.24             11.2              0.0
               0.50         0.10422     34.739        4.42     2.93              2.5              0.0
               1.00         0.26826     77.776        5.25     3.33              0.0              0.0
               2.50         0.76219     78.475       12.58     8.23              0.0              0.0
               5.00         1.68241     81.493       25.91    15.18              0.0              0.0
               7.50         2.74806     87.488       40.51    25.65              0.0              0.0
              10.00         4.02595     97.528       50.00    34.50              0.0              0.0
              15.00         6.10341    118.43